# Semana 10 · Proyecto 1 · Cadena de señal TMP36

Cuaderno de diseño **nominal/simulado**. No sustituye la caracterización experimental de TMP36, amplificador y ADC del ESP32.

## 1. Especificaciones del instrumento

Intervalo 20–120 °C, resolución objetivo 0.05 °C, una muestra por segundo y dos límites configurables. Acondicionamiento analógico: **restar 0.5 V y amplificar**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
R1, R2 = 10_000, 24_900
G = R2/R1
V_OFF_SENSOR = 0.5
V_REF_RESTA = 0.5
S = 0.010  # V/°C
T = np.linspace(20,120,1001)
Vs = V_OFF_SENSOR + S*T
Vestrella = Vs - V_REF_RESTA
Vo = G*Vestrella
D_ideal = np.rint(np.clip(Vo/3.3,0,1)*4095).astype(int)
print('G =',G)
for i in (0,-1):print(f'T={T[i]:.0f}°C | Vs={Vs[i]:.3f}V | V*={Vestrella[i]:.3f}V | Vo={Vo[i]:.3f}V | D ideal={D_ideal[i]}')


## 2. Gráficas con ejes numerados y unidades

Observe cómo se obtiene el margen de salida de cada etapa.

In [ ]:
for curve, name, unit in [(Vs,'TMP36 Vs','V'),(Vestrella,'Salida restador V*','V'),(Vo,'Salida amplificador Vo','V'),(D_ideal,'Código ADC ideal','cuentas')]:
    plt.figure(figsize=(7.0,3.8))
    plt.plot(T,curve)
    plt.xlabel('Temperatura T (°C)')
    plt.ylabel(f'{name} ({unit})')
    plt.xticks(np.arange(20,121,20))
    plt.grid(True,alpha=.35)
    plt.tight_layout()
    plt.show()


## 3. Inversión de la cadena de señal

La fórmula correcta incluye el valor **real** de la referencia restada y el offset nominal/calibrado del sensor.

In [ ]:
T_rec = (Vo/G + V_REF_RESTA - V_OFF_SENSOR)/S
assert np.max(np.abs(T-T_rec)) < 1e-9
print('Error numérico nominal máximo (°C):', np.max(np.abs(T-T_rec)))
q = 3.3/4096
print(f'LSB nominal = {q*1e3:.3f} mV; resolución térmica ideal ≈ {q/(G*S):.4f} °C')
print('Es un resultado ideal y NO describe la exactitud del TMP36 ni el ruido efectivo del ESP32.')


## 4. Histéresis y tres estados

Adapte los límites inferior/superior y verifique que calefacción y enfriamiento nunca estén encendidos simultáneamente.

In [ ]:
lo, hi, h = 35., 45., 1.0
state='APAGADO'
def step(temp, state):
    if state=='CALEFACCION' and temp < lo+h: return state
    if state=='ENFRIAMIENTO' and temp > hi-h: return state
    if temp<lo:return 'CALEFACCION'
    if temp>hi:return 'ENFRIAMIENTO'
    return 'APAGADO'
for temp in [40,30,35.5,36,50,44.5,44,40]:
    state=step(temp,state)
    print(temp,state)


## 5. Evidencia de laboratorio

Registre datos reales desde la GUI Python, expórtelos a CSV e impórtelos aquí. Indique la calibración, incertidumbre, resolución y desviaciones. No presente valores simulados como experimentales.

In [ ]:
# Archivo de la GUI: Proyecto1_TMP36_mediciones.csv
# Descomente las siguientes líneas cuando disponga de datos experimentales:
# import pandas as pd
# df = pd.read_csv('Proyecto1_TMP36_mediciones.csv')
# print(df.head())
# print(df['temperatura_c'].agg(['mean','std','min','max']))
